In [8]:
import os

#直接使用GPT2模型来计算奖励,输入是一个文本序列，输出是该序列的奖励值。gpt+线性层
import torch
from sympy import false
from torch import nn
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer



class RewardHead(nn.Module):
    def __init__(self, config):
        super().__init__()
        # llm最后输出的隐藏层的维度
        self.hidden_size = config.hidden_size
        # 线性层用来对llm最后输出的隐藏层给奖励
        self.reward = nn.Linear(self.hidden_size, 1)
        self._post_init()

    def _post_init(self):
        # 使用正态分布初始化权重
        nn.init.normal_(
            self.reward.weight,
            std=(1.0 / np.sqrt(self.hidden_size + 1))
        )
        # 将偏置初始化为0
        nn.init.zeros_(self.reward.bias)

    def forward(self, hidden_states):
        # 给出奖励
        return self.reward(hidden_states)

class GPT2RewardHead(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.llm = AutoModelForCausalLM.from_pretrained(model_name)
        self.reward_head = RewardHead(self.llm.config)
        '''第二种只修改头'''
        # 冻结 GPT-2 的所有参数
        # for param in self.llm.parameters():
        #     param.requires_grad = False
        # # 将 GPT-2 设置为评估模式（影响 dropout 和 batchnorm）
        # self.llm.eval()

        '''第三种只修改最后两三层'''
         # 解冻最后两层 transformer 块和最后的 LayerNorm
        for layer in self.llm.transformer.h[:-2]:
            for param in layer.parameters():
                param.requires_grad = False



    def forward(self, input_ids, attention_mask):
        transformer_outputs = self.llm.forward(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )
        last_hidden_state = transformer_outputs.hidden_states[-1] # 获取最后一层的隐藏状态，而不是输出的logits

        #第一种
        # # 给出奖励
        # reward = self.reward_head(last_hidden_state).squeeze(-1)
        # # sigmoid用来将奖励搞到(-1,1)范围内
        # return torch.sigmoid(reward)

        #第二种
        # 平均池化
        # 注意：需要根据 attention_mask 计算平均，忽略 padding 部分
        mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
        sum_embeddings = torch.sum(last_hidden_state * mask_expanded, dim=1)
        sum_mask = torch.clamp(mask_expanded.sum(dim=1), min=1e-9)
        pooled = sum_embeddings / sum_mask  # (batch, hidden)
        reward = self.reward_head(pooled)  # (batch, 1)
        return reward.squeeze(-1)  # (batch, )  # 返回 logits，不是 sigmoid

model_path = "C:\\Users\lihaodong\.cache\modelscope\hub\models\Fengshenbang\Wenzhong-GPT2-110M-chinese-v2"
model = GPT2RewardHead(model_path)
if os.path.exists('models/best_reward_model.pt'):
    model.load_state_dict(torch.load('models/best_reward_model.pt'))
    print('reward_model.pt loaded')


In [9]:
from functools import partial
#加载tokenizer
from datasets import load_dataset
from transformers import AutoTokenizer


#定义collate
def reward_collate_fn(batch):
    # batch 是一个列表，每个元素是字典，包含 'input_ids', 'attention_mask', 'score', 'score_index' 等
    input_ids = [item['input_ids'] for item in batch]
    attention_mask = [item['attention_mask'] for item in batch]
    scores = [item['score'] for item in batch]
    # 原有的 score_index 我们不再使用，直接重新计算：
    # 每个样本的 reward token 在原始序列中的位置就是最后一个 token
    # 但经过 padding 后，我们需要找到每个样本的最后一个有效 token 的位置
    # 方法：先 padding，然后根据 attention_mask 找到每个样本的最后一个 1 的位置
    # 注意：padding 会改变序列长度，但我们可以在 padding 后再计算。

    # 使用 tokenizer 进行 padding
    padded = tokenizer.pad(
        {'input_ids': input_ids, 'attention_mask': attention_mask},
        padding=True,
        return_tensors='pt'
    )

    # 计算每个样本的最后一个有效 token 的索引
    # attention_mask 是 (batch_size, seq_len)，每行是 0/1
    # 使用 torch.argmax 可以找到最后一个 1 的位置（因为 argmax 返回第一个最大值的索引，对于 0/1 序列，最后一个 1 的位置需要更复杂的方法）
    # 简便方法：取每行非零位置的最大索引
    seq_len = padded['attention_mask'].shape[1]
    # 创建一个索引矩阵
    indices = torch.arange(seq_len).unsqueeze(0).expand(padded['attention_mask'].shape[0], -1).to(padded['attention_mask'].device)
    # 将 attention_mask 为 0 的位置索引设为 -1
    masked_indices = indices * padded['attention_mask']
    # 取每行的最大值，就是最后一个有效 token 的索引
    score_index = masked_indices.max(dim=1).values

    # 将 scores 转换为 tensor
    scores_tensor = torch.tensor(scores, dtype=torch.float32)

    # 返回一个字典，包含 input_ids, attention_mask, score, score_index
    return {
        'input_ids': padded['input_ids'],
        'attention_mask': padded['attention_mask'],
        'score': scores_tensor,
        'score_index': score_index
    }

# 定义分词函数：接收一个 batch 字典，返回一个字典（键是新的列名，值是列表）
def tokenize(batch, tokenizer, REWARD_TOKEN_ID):
    # 分词
    contexts = tokenizer(batch['context'])
    batch_size = len(contexts['input_ids'])  # 批次大小

    # 初始化得分和得分位置（长度 = batch_size）
    contexts["score"] = [0.0] * batch_size
    contexts["score_index"] = [0] * batch_size

    for i in range(batch_size):
        # 在每条文本末尾添加 reward token（eos token）
        contexts['input_ids'][i].append(REWARD_TOKEN_ID)
        contexts['attention_mask'][i].append(1)

        # 评分：label 是 1（正向）或 0（负向）
        contexts['score'][i] = float(batch['label'][i])

        # 记录 reward token 的索引（最后一个位置）
        contexts['score_index'][i] = len(contexts['input_ids'][i]) - 1

    return contexts


#清理数据长度国小的
def filter_short_samples(batch, min_length=6):
    # 计算输入 ID 的长度
    input_ids_length = len(batch['input_ids'])
    # 只有当长度大于或等于 min_length和小于等于100 时才保留样本
    return (input_ids_length >= min_length and input_ids_length <= 150)

def load_data(data_path,tokenizer):
    dataset = load_dataset('json', data_files={
        'train': data_path[0],
        'valid': data_path[1],
        'test': data_path[2]
    })
    # 查看结构
    print(dataset['train'][0])
    # 输出：{'idx': 0, 'context': '如果 我 无聊 时 ...', 'label': 1}

    map_kwargs = {
        'batched': True,  # 启用批处理模式，tokenize 接收的是 batch 字典
        'batch_size': 512,  # 每批 512 个样本
        'remove_columns': ['idx', 'context', 'label']  # 处理后移除这三列
    }
    REWARD_TOKEN_ID = tokenizer.eos_token_id
    print(REWARD_TOKEN_ID)
    # 在 map 时绑定 tokenizer
    tokenize_with_tokenizer = partial(tokenize, tokenizer=tokenizer,REWARD_TOKEN_ID = REWARD_TOKEN_ID)

    # 对训练集和验证集应用 map
    tokenized_dataset_train = dataset["train"].map(tokenize_with_tokenizer, **map_kwargs)
    tokenized_dataset_val = dataset["valid"].map(tokenize_with_tokenizer, **map_kwargs)

    print(tokenized_dataset_train[0])
    print(tokenized_dataset_val[0])


    # 对训练集和验证集应用过滤函数
    filtered_dataset_train = tokenized_dataset_train.filter(filter_short_samples)
    filtered_dataset_val = tokenized_dataset_val.filter(filter_short_samples)
    print(f"训练集过滤前样本数量: {len(tokenized_dataset_train)}")
    print(f"训练集过滤后样本数量: {len(filtered_dataset_train)}")
    print(f"验证集过滤前样本数量: {len(tokenized_dataset_val)}")
    print(f"验证集过滤后样本数量: {len(filtered_dataset_val)}")

    # 放入torch
    import torch
    filtered_dataset_train.set_format(type='torch')
    filtered_dataset_val.set_format(type='torch')
    print(filtered_dataset_train[0])
    print(filtered_dataset_val[0])


    return filtered_dataset_train, filtered_dataset_val, dataset["test"]

data_path = ["../SFT微调大模型/data_raw/train.jsonl","../SFT微调大模型/data_raw/valid.jsonl","../SFT微调大模型/data_raw/test.jsonl"]
tokenizer = AutoTokenizer.from_pretrained(model_path)
filtered_dataset_train, filtered_dataset_val, dataset_test = load_data(data_path=data_path,tokenizer=tokenizer)

{'idx': 0, 'context': '死囚爱刽子手女贼爱衙役我们爱你们难道还有别的选择没想到胡军除了蓝宇还有东宫西宫我个去阿兰这样真他nia恶心爱个P分明只是欲', 'label': 1}
50256
{'input_ids': [29826, 119, 32368, 248, 163, 230, 109, 26344, 121, 36310, 33699, 233, 42637, 164, 112, 120, 163, 230, 109, 26193, 247, 37605, 117, 22755, 239, 20015, 105, 163, 230, 109, 19526, 254, 20015, 105, 49694, 122, 34402, 241, 32573, 246, 17312, 231, 26344, 104, 21410, 34460, 231, 162, 233, 102, 162, 110, 94, 46349, 111, 26344, 108, 47797, 94, 37863, 249, 165, 247, 97, 12859, 228, 164, 241, 251, 22522, 229, 32573, 246, 17312, 231, 10310, 250, 22522, 104, 164, 98, 123, 22522, 104, 22755, 239, 10310, 103, 43889, 119, 165, 246, 123, 17739, 108, 32573, 247, 43718, 115, 40367, 253, 20015, 244, 18142, 162, 223, 114, 33232, 225, 163, 230, 109, 10310, 103, 47, 26344, 228, 23626, 236, 20998, 103, 42468, 162, 105, 110, 50256], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,

In [10]:
from torch.utils.data import DataLoader
from transformers import DataCollatorWithPadding
# 将 eos token 作为 pad token
tokenizer.pad_token = tokenizer.eos_token

# data_collator = DataCollatorWithPadding(tokenizer)
dataloader_params = {
    'batch_size': 16,
    'shuffle': True,
    'collate_fn': reward_collate_fn
}
train_dataloader = DataLoader(filtered_dataset_train, **dataloader_params)
val_dataloader = DataLoader(filtered_dataset_val, **dataloader_params)

batch = next(iter(train_dataloader))
print(batch.keys())

print(batch['input_ids'][1])
print(batch['attention_mask'][1])
print(batch['score'][1])
print(batch['score_index'][1])
print(tokenizer.decode(batch['input_ids'][1]))
print(batch['attention_mask'][1].nonzero()[-1])

You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


dict_keys(['input_ids', 'attention_mask', 'score', 'score_index'])
tensor([22755,   239, 17312,   105, 20015,    98, 10310,   118, 27670,   248,
          164,   122,   122, 26344,   108, 25001,   121,   164,   236,   109,
          161,   251,   252, 21689, 31965,   102, 27670,   254,   164,   106,
          108, 18796,   113, 37605,   109,   165, 45865, 41753,    99, 20998,
          107, 39355,   230, 13783,   250,   162,    95,    99, 32368,   252,
        40792, 22755,   239, 20998,   239,   163,   236,   108, 22755,   239,
        37605,   119, 41753,   243,   165,   242,   247, 26193,   101,   162,
          120,   242, 36181,   230, 25001,   121, 19526,   228, 31965,   229,
        36310,   163,   119,   241,   162,   252,   226, 30298,   103,   164,
          122,   239, 43095, 28156,   237, 29785,   106,   165,    95,   246,
        32368,   121, 12859,   100, 18796,   113, 37605,   109, 47797,   121,
        26344,   104, 37863,   235,   162,   242,   122, 33176,   119,   16

In [11]:
outputs = model(batch['input_ids'], batch['attention_mask'])
print(outputs.shape)
print(outputs)
print(batch['score_index'])
print(batch['score'])

torch.Size([16])
tensor([-0.9485, -0.8235, -0.4626, -0.3401, -0.6973, -0.3612, -0.5002, -1.0203,
        -0.3513, -0.7374, -0.6841, -0.8329, -0.4965, -0.7257, -0.2479, -0.5177],
       grad_fn=<SqueezeBackward1>)
tensor([142, 126, 148, 124, 131, 130, 145, 122, 122, 119, 148, 127, 144, 142,
        102, 142])
tensor([1., 1., 1., 1., 1., 0., 1., 1., 0., 0., 0., 1., 0., 1., 0., 1.])


In [12]:
#论文里面是只训练了一个epoch，奖励模型的训练通常需要较少的epoch，因为它们通常是用来评估生成模型的输出，而不是直接生成文本。过多的训练可能会导致奖励模型过拟合，从而无法有效地评估生成模型的输出。

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)
# 二分类交叉熵损失
criterion = nn.BCEWithLogitsLoss()
num_epochs = 1


In [15]:
#评估
# def validate():
#     model.eval()
#     total_loss = 0
#     for i, batch in enumerate(val_dataloader):
#         # 将 tensor 字段转移到设备
#         input_ids = batch['input_ids'].to(device)
#         attention_mask = batch['attention_mask'].to(device)
#         score_index = batch['score_index'].to(device)
#         target = batch['score'].to(device)
#         model_inputs = {
#             'input_ids': input_ids,
#             'attention_mask': attention_mask
#         }
#         with torch.no_grad():
#             # 对输出进行评分
#             scores = model(**model_inputs)
#             # 批次中每条数据的索引
#             batch_indices = torch.arange(scores.shape[0])
#             # 根据索引拿出评分，也就是reward token的评分
#             score = scores[batch_indices, score_index]
#             # 目标评分，0 或者 1 。
#             target = target  #需要注意的是这里的评分都是大于0的，但是可以通过调整reward token的位置来让模型学会给负向样本打分更低，给正向样本打分更高。
#             # 计算误差
#             loss = criterion(score, target)
#         total_loss += loss.item()
#     print('validation loss:', total_loss / len(val_dataloader))

#第二种
def validate():
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for batch in val_dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            target = batch['score'].to(device)          # 注意：不再使用 score_index
            scores = model(input_ids, attention_mask)   # (batch,)
            loss = criterion(scores, target)
            total_loss += loss.item()
    avg_loss = total_loss / len(val_dataloader)
    print(f'Validation loss: {avg_loss:.4f}')
    return avg_loss
import os
if os.path.exists('models/best_reward_model.pt'):
    model.load_state_dict(torch.load('models/best_reward_model.pt'))
    print('reward_model.pt loaded')
model.to(device)

validate()



Validation loss: 0.5835


0.5835332679562271

In [18]:
#训练
# 加载已有的模型

# for epoch in range(num_epochs):
#     model.train()
#     for i, batch in enumerate(train_dataloader):
#          # 将 tensor 字段转移到设备
#         input_ids = batch['input_ids'].to(device)
#         attention_mask = batch['attention_mask'].to(device)
#         score_index = batch['score_index'].to(device)
#         target = batch['score'].to(device)
#
#         model_inputs = {
#             'input_ids': input_ids,
#             'attention_mask': attention_mask
#         }
#         scores = model(**model_inputs)
#         batch_indices = torch.arange(scores.shape[0])
#         score = scores[batch_indices, score_index]
#         target = target
#         loss = criterion(score, target)
#         #清空梯度 ⟶ 反向传播计算梯度 ⟶ 更新参数
#         optimizer.zero_grad()
#         loss.backward()
#         optimizer.step()
#         print(loss.item())

#第二种
if os.path.exists('models/best_reward_model.pt'):
    model.load_state_dict(torch.load('models/best_reward_model.pt'))
    print('reward_model.pt loaded')
validate()

best_val_loss = float('inf')
for epoch in range(num_epochs):
    model.train()
    for i, batch in enumerate(train_dataloader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        target = batch['score'].to(device)           # 注意：不再使用 score_index

        scores = model(input_ids, attention_mask)    # (batch,)
        loss = criterion(scores, target)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # 每 50 步打印一次训练 loss 并验证
        if i % 50 == 0:
            print(f'Epoch {epoch}, step {i}, train loss: {loss.item():.4f}')
            val_loss = validate()
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                torch.save(model.state_dict(), 'models/best_reward_model.pt')
                print(f'Best model saved with val loss {val_loss:.4f}')

    # 每个 epoch 结束验证一次
    val_loss = validate()
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'models/best_reward_model.pt')


reward_model.pt loaded
Validation loss: 0.5118
Epoch 0, step 0, train loss: 0.2935
Validation loss: 0.5103
Best model saved with val loss 0.5103
Epoch 0, step 50, train loss: 0.3752
Validation loss: 0.5095
Best model saved with val loss 0.5095
Epoch 0, step 100, train loss: 0.2949
Validation loss: 0.5063
Best model saved with val loss 0.5063
Epoch 0, step 150, train loss: 0.5450
Validation loss: 0.5081
Epoch 0, step 200, train loss: 0.3285
Validation loss: 0.5113
Epoch 0, step 250, train loss: 0.3560
Validation loss: 0.5361
Epoch 0, step 300, train loss: 0.5958
Validation loss: 0.5183
Epoch 0, step 350, train loss: 0.3071
Validation loss: 0.5089
Epoch 0, step 400, train loss: 0.5621
Validation loss: 0.5138
Epoch 0, step 450, train loss: 0.3602
Validation loss: 0.5041
Best model saved with val loss 0.5041
Validation loss: 0.5038


In [22]:
# torch.save(model.state_dict(), 'models/reward_model.pt')
# from sklearn.metrics import confusion_matrix
# model.to(device)
# model.eval()
#
# all_predictions = []
# all_labels = []
#
# for i, batch in enumerate(val_dataloader):
#     input_ids = batch['input_ids'].to(device)
#     attention_mask = batch['attention_mask'].to(device)
#     score_index = batch['score_index'].to(device)
#     target = batch['score'].to(device)
#     model_inputs = {
#         'input_ids': input_ids,
#         'attention_mask':attention_mask
#     }
#     with torch.no_grad():
#         scores = model(**model_inputs)
#         batch_indices = torch.arange(scores.shape[0])
#         score = scores[batch_indices, score_index]
#         target = target
#     predictions = (score > 0.5).int()
#
#     all_predictions.extend(predictions.cpu().numpy())
#     all_labels.extend(target.cpu().numpy())
#
# confusion_matrix(all_labels, all_predictions)
#困惑矩阵:
# 第一行：真实标签为0的情况：756个预测正确，275个预测错误
 # 第二行：真实标签为1的情况：234个预测错误，772个预测正确

#第二种
from sklearn.metrics import confusion_matrix
model.to(device)
model.eval()

all_predictions = []
all_labels = []

for batch in val_dataloader:
    input_ids = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    target = batch['score'].to(device)        # 直接使用 target，不再需要 score_index

    with torch.no_grad():
        scores = model(input_ids, attention_mask)   # scores 是一维 (batch_size,)
        probs = torch.sigmoid(scores)              # 转为概率 (0~1)
    predictions = (probs > 0.5).int()

    all_predictions.extend(predictions.cpu().numpy())
    all_labels.extend(target.cpu().numpy())

print(confusion_matrix(all_labels, all_predictions))

[[800 231]
 [248 758]]


In [21]:
#评估
#加载已有的模型
# import os
# if os.path.exists('models/reward_model.pt'):
#     model.load_state_dict(torch.load('models/reward_model.pt'))
#     print('reward_model.pt loaded')
model.to(device)
validate()
print(model)

KeyboardInterrupt: 